In [14]:
%pip install fairlearn


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: C:\Users\Oluwadarasimi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
print(os.getcwd())

c:\Users\Oluwadarasimi\pre-phd\bias-recruitment-testing\notebooks


In [3]:
import numpy as np

data = np.load('../data/FairCVtest/data/FairCVdb.npy', allow_pickle=True)
print(type(data))
print(data.shape if hasattr(data, 'shape') else len(data))

<class 'numpy.ndarray'>
()


In [4]:
data_dict = data.item()
print(type(data_dict))
print(data_dict.keys() if hasattr(data_dict, 'keys') else data_dict)

<class 'dict'>
dict_keys(['Profiles Train', 'Profiles Test', 'Bios Train', 'Bios Test', 'Names Train', 'Names Test', 'Blind Labels Train', 'Blind Labels Test', 'Biased Labels Train (Gender)', 'Biased Labels Test (Gender)', 'Biased Labels Train (Ethnicity)', 'Biased Labels Test (Ethnicity)', 'Image List Train', 'Image List Test'])


load test sets

In [5]:
print("Profiles Train:", data_dict['Profiles Train'].shape)
print("Blind Labels Train:", data_dict['Blind Labels Train'].shape)
print("Biased Labels Train (Gender):", data_dict['Biased Labels Train (Gender)'].shape)
print("Biased Labels Train (Ethnicity):", data_dict['Biased Labels Train (Ethnicity)'].shape)

Profiles Train: (19200, 51)
Blind Labels Train: (19200,)
Biased Labels Train (Gender): (19200,)
Biased Labels Train (Ethnicity): (19200,)


In [6]:
print("Profiles Test:", data_dict['Profiles Test'].shape)
print("Blind Labels Test:", data_dict['Blind Labels Test'].shape)
print("Biased Labels Test (Gender):", data_dict['Biased Labels Test (Gender)'].shape)
print("Biased Labels Test (Ethnicity):", data_dict['Biased Labels Test (Ethnicity)'].shape)

Profiles Test: (4800, 51)
Blind Labels Test: (4800,)
Biased Labels Test (Gender): (4800,)
Biased Labels Test (Ethnicity): (4800,)


In [7]:
print(data_dict['Profiles Train'][0])

[ 1.          0.          4.          0.25        0.6         0.8
  0.          1.          0.6         1.          0.4         0.27422652
  0.04474127 -0.34303778  0.19849196  0.18480811 -0.17857905  0.2952157
  0.13379891  0.43267974 -0.097774    0.25182572 -0.0931767   0.27867725
 -0.07276464 -0.08267434 -0.05022223  0.31611171  0.09124116 -0.14106941
  0.32448468 -0.28245971  0.20202184  0.26307192  0.19638553  0.1544019
  0.30405623 -0.18988736  0.22052124  0.1836338  -0.22950351  0.21591355
 -0.11140636  0.14146559 -0.44647449  0.20560384 -0.16441558 -0.14390801
  0.13059542 -0.17022148 -0.26425746]


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

X_train = data_dict['Profiles Train']
y_train_biased = data_dict['Biased Labels Train (Gender)']

X_test = data_dict['Profiles Test']
y_test_biased = data_dict['Biased Labels Test (Gender)']
y_test_blind = data_dict['Blind Labels Test']

# Train on the biased labels — this simulates the "biased AI recruiter"
model = LogisticRegression(max_iter=1000)
model.fit(X_train, (y_train_biased > 0.5).astype(int))  # binarize if labels are continuous scores

# Check predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy vs biased test labels:", accuracy_score((y_test_biased > 0.5).astype(int), y_pred))
print("Accuracy vs BLIND (fair) test labels:", accuracy_score((y_test_blind > 0.5).astype(int), y_pred))


Accuracy vs biased test labels: 0.9766666666666667
Accuracy vs BLIND (fair) test labels: 0.9052083333333333


In [9]:
import joblib
import os

os.makedirs('../src/models', exist_ok=True)
joblib.dump(model, '../src/models/baseline_biased_recruiter.pkl')
print("Saved.")

Saved.


In [11]:
import sys
print(sys.executable)

C:\Users\Oluwadarasimi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [15]:
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
from sklearn.metrics import accuracy_score

# Sensitive feature = gender, index 1 in the profile vector
gender_test = X_test[:, 1]  # 0 = Male, 1 = Female

# Use the model's actual predictions (already computed as y_pred)
y_true_blind = (y_test_blind > 0.5).astype(int)  # fair ground truth

# Statistical parity: does the model select men/women at different rates?
dp_diff = demographic_parity_difference(
    y_true_blind, y_pred, sensitive_features=gender_test
)

# Equalized odds: do error rates (FPR/FNR) differ by gender?
eo_diff = equalized_odds_difference(
    y_true_blind, y_pred, sensitive_features=gender_test
)

print("Demographic parity difference:", dp_diff)
print("Equalized odds difference:", eo_diff)

# Breakdown by group for context
metric_frame = MetricFrame(
    metrics=accuracy_score,
    y_true=y_true_blind,
    y_pred=y_pred,
    sensitive_features=gender_test
)
print("\nAccuracy by gender group:")
print(metric_frame.by_group)

Demographic parity difference: 0.1458070503215087
Equalized odds difference: 0.6073252988146605

Accuracy by gender group:
sensitive_feature_0
0.0    0.973738
1.0    0.834532
Name: accuracy_score, dtype: float64


In [16]:
# Train a second baseline model on ethnicity-biased labels
y_train_biased_eth = data_dict['Biased Labels Train (Ethnicity)']
y_test_biased_eth = data_dict['Biased Labels Test (Ethnicity)']

model_eth = LogisticRegression(max_iter=1000)
model_eth.fit(X_train, (y_train_biased_eth > 0.5).astype(int))

y_pred_eth = model_eth.predict(X_test)

ethnicity_test = X_test[:, 0]  # index 0 = ethnicity (G1/G2/G3)

dp_diff_eth = demographic_parity_difference(
    y_true_blind, y_pred_eth, sensitive_features=ethnicity_test
)
eo_diff_eth = equalized_odds_difference(
    y_true_blind, y_pred_eth, sensitive_features=ethnicity_test
)

print("Demographic parity difference (ethnicity):", dp_diff_eth)
print("Equalized odds difference (ethnicity):", eo_diff_eth)

metric_frame_eth = MetricFrame(
    metrics=accuracy_score,
    y_true=y_true_blind,
    y_pred=y_pred_eth,
    sensitive_features=ethnicity_test
)
print("\nAccuracy by ethnicity group:")
print(metric_frame_eth.by_group)

Demographic parity difference (ethnicity): 0.4499776085982983
Equalized odds difference (ethnicity): 0.6594594594594594

Accuracy by ethnicity group:
sensitive_feature_0
0.0    0.724765
1.0    0.972292
2.0    0.849103
Name: accuracy_score, dtype: float64


In [17]:
joblib.dump(model_eth, '../src/models/baseline_biased_recruiter_ethnicity.pkl')
print("Saved.")

Saved.
